# APAC Risk Monitor — Week 1 exploration

Config-driven ingestion, cleaning, and **static-weight** portfolio returns.
Instrument names and weights come only from `config/portfolio.yaml`.

Run this notebook from the repository root (or with that directory on `PYTHONPATH`).

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_loader import load_portfolio_config
from src.pipeline import run_week1_pipeline

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
cfg = load_portfolio_config()
print(cfg.portfolio_name)
print(f"Base currency: {cfg.base_currency}  |  Capital: {cfg.total_capital:,.0f}  |  Benchmark: {cfg.benchmark}")
composition = pd.DataFrame(
    [
        {
            "ticker": p.ticker,
            "sleeve": p.sleeve,
            "weight": p.weight,
            "rationale": p.rationale,
        }
        for p in cfg.positions
    ]
)
composition

## End-to-end pipeline

Downloads ~5 years of adjusted closes, writes dated snapshots, cleans the panel, and computes portfolio returns. Re-running creates a new dated file rather than overwriting.

In [ ]:
result = run_week1_pipeline()
print("Raw snapshot:", result.ingestion.snapshot_path)
print("Returns:", result.cleaning.returns_path)
print("Portfolio:", result.portfolio.output_path)
print("\nCleaning decisions")
for line in result.cleaning.decisions:
    print(" -", line)

## Cumulative performance vs benchmark

In [ ]:
import matplotlib.pyplot as plt

perf = result.portfolio.returns[["cumulative_portfolio", "cumulative_benchmark"]].copy()
perf.columns = [result.config.portfolio_name, f"Benchmark ({result.config.benchmark})"]

fig, ax = plt.subplots(figsize=(10, 4.5))
perf.plot(ax=ax, linewidth=1.4)
ax.set_title("Cumulative wealth (static weights, start = 1.0)")
ax.set_ylabel("Growth of $1")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

perf.tail()

## Validation and warnings

In [ ]:
print("Validation")
for line in result.validation.summary_lines():
    print(" ", line)

print("\nIngestion warnings")
if result.ingestion.warnings:
    for w in result.ingestion.warnings:
        print(" -", w)
else:
    print(" (none)")

flags = result.cleaning.extreme_flags_frame()
print("\nExtreme-return flags (|r| > 10%)")
flags if not flags.empty else "(none)"